In [2]:
%pip install plotly pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 14.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]
Note: you may need to restart the kernel to use updated packages.


In [7]:
from src.data import load_train
import numpy as np
import pandas as pd
import sys
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import plot
from IPython.display import HTML, display
from pathlib import Path

# Substitua caso seu DataFrame tenha outro nome
X, y = load_train()
dados = X.copy()
dados["stars"] = y.to_numpy()

# Preparação
cols = [
    "user_avg_stars", "user_n_ratings", "age",
    "share_action", "share_comedy", "share_drama",
    "movie_avg_stars", "movie_n_ratings",
    "genre", "release_year", "stars"
]
dados = dados[cols].copy()

for c in cols:
    if c != "genre":
        dados[c] = pd.to_numeric(dados[c], errors="coerce")

dados["genre"] = dados["genre"].astype("string")
dados = dados.dropna(subset=[
    "user_avg_stars", "movie_avg_stars", "stars"
])

template = "plotly_dark"

# --------------------------------------------------
# 1. SCATTER 3D
# --------------------------------------------------

# Amostragem para não sobrecarregar o navegador
amostra = dados.sample(
    n=min(12000, len(dados)),
    random_state=42
).copy()

fig1 = px.scatter_3d(
    amostra,
    x="user_avg_stars",
    y="movie_avg_stars",
    z="stars",
    color="genre",
    size="movie_n_ratings",
    size_max=10,
    opacity=0.55,
    hover_data=[
        "age", "release_year",
        "user_n_ratings", "movie_n_ratings"
    ],
    title="3D | Perfil do usuário × Reputação do filme × Nota",
    template=template
)

fig1.update_layout(
    height=720,
    scene=dict(
        xaxis_title="Média do usuário",
        yaxis_title="Média do filme",
        zaxis_title="Nota atribuída"
    )
)

# --------------------------------------------------
# 2. SUPERFÍCIE 3D: NOTA MÉDIA OBSERVADA
# --------------------------------------------------

base = dados.dropna(subset=[
    "user_avg_stars", "movie_avg_stars", "stars"
]).copy()

bins = np.linspace(0, 5, 21)
centros = (bins[:-1] + bins[1:]) / 2

base["user_bin"] = pd.cut(
    base["user_avg_stars"],
    bins=bins, labels=centros, include_lowest=True
)
base["movie_bin"] = pd.cut(
    base["movie_avg_stars"],
    bins=bins, labels=centros, include_lowest=True
)

pivot = base.pivot_table(
    index="movie_bin",
    columns="user_bin",
    values="stars",
    aggfunc="mean",
    observed=False
)

contagens = base.pivot_table(
    index="movie_bin",
    columns="user_bin",
    values="stars",
    aggfunc="count",
    observed=False
)

# Esconde células com poucas observações
z = pivot.where(contagens >= 10).to_numpy(dtype=float)

fig2 = go.Figure(
    data=go.Surface(
        x=centros,
        y=centros,
        z=z,
        colorscale="Viridis",
        colorbar=dict(title="Nota média")
    )
)

fig2.update_layout(
    title="Superfície 3D | Nota média observada",
    template=template,
    height=700,
    scene=dict(
        xaxis_title="Média do usuário",
        yaxis_title="Média do filme",
        zaxis_title="Nota média real",
        zaxis=dict(range=[0, 5])
    )
)

# --------------------------------------------------
# 3. HEATMAP COM OS MESMOS DADOS DA SUPERFÍCIE
# --------------------------------------------------

fig3 = go.Figure(
    go.Heatmap(
        x=centros,
        y=centros,
        z=z,
        colorscale="Viridis",
        zmin=0,
        zmax=5,
        colorbar=dict(title="Nota média"),
        customdata=contagens.to_numpy(),
        hovertemplate=(
            "Média usuário: %{x:.2f}<br>"
            "Média filme: %{y:.2f}<br>"
            "Nota observada: %{z:.2f}<br>"
            "Avaliações na célula: %{customdata}<extra></extra>"
        )
    )
)

fig3.update_layout(
    title="Heatmap | Interação entre usuário e filme",
    xaxis_title="Média do usuário",
    yaxis_title="Média do filme",
    template=template,
    height=600
)

# --------------------------------------------------
# 4. NOTAS POR GÊNERO E FAIXA ETÁRIA
# --------------------------------------------------

base_idade = dados.dropna(
    subset=["age", "genre", "stars"]
).copy()

base_idade["faixa_etaria"] = pd.cut(
    base_idade["age"],
    bins=[15, 25, 35, 45, 55, 65, 81],
    labels=["16–25", "26–35", "36–45",
            "46–55", "56–65", "66–80"],
    right=False
)

fig4 = px.violin(
    base_idade.sample(
        n=min(25000, len(base_idade)),
        random_state=42
    ),
    x="faixa_etaria",
    y="stars",
    color="genre",
    box=True,
    points=False,
    title="Distribuição das notas | Idade × Gênero",
    template=template
)

fig4.update_layout(
    height=600,
    xaxis_title="Faixa etária",
    yaxis_title="Nota"
)

# --------------------------------------------------
# 5. PREFERÊNCIA POR GÊNERO × NOTA
# --------------------------------------------------

pref = dados.copy()

pref["genero_preferido"] = (
    pref[["share_action", "share_comedy", "share_drama"]]
    .idxmax(axis=1)
    .str.replace("share_", "", regex=False)
)

cruzamento = (
    pref.dropna(subset=["genero_preferido", "genre", "stars"])
    .groupby(["genero_preferido", "genre"])
    .agg(
        nota_media=("stars", "mean"),
        quantidade=("stars", "size")
    )
    .reset_index()
)

fig5 = px.density_heatmap(
    pref,
    x="genero_preferido",
    y="genre",
    z="stars",
    histfunc="avg",
    text_auto=".2f",
    color_continuous_scale="Viridis",
    title="Nota média | Gênero preferido × Gênero do filme",
    template=template
)

fig5.update_layout(
    height=550,
    xaxis_title="Preferência dominante do usuário",
    yaxis_title="Gênero do filme"
)

# --------------------------------------------------
# 6. DISTRIBUIÇÃO GERAL DAS ESTRELAS
# --------------------------------------------------

fig6 = px.histogram(
    dados,
    x="stars",
    color="genre",
    barmode="group",
    histnorm="percent",
    title="Distribuição de notas por gênero",
    template=template
)

fig6.update_layout(
    height=550,
    xaxis_title="Estrelas",
    yaxis_title="Percentual"
)

# --------------------------------------------------
# EXPORTAÇÃO: UM ÚNICO ARQUIVO HTML
# --------------------------------------------------

graficos = [fig1, fig2, fig3, fig4, fig5, fig6]

titulos = [
    "Exploração 3D",
    "Superfície 3D",
    "Heatmap",
    "Idade e gênero",
    "Preferências",
    "Distribuição"
]

partes = []

for i, (fig, titulo) in enumerate(zip(graficos, titulos)):
    partes.append(
        f'<section class="grafico"><h2>{titulo}</h2>'
        + plot(
            fig,
            output_type="div",
            include_plotlyjs=(True if i == 0 else False),
            config={"responsive": True, "displaylogo": False}
        )
        + "</section>"
    )

html = """
<!DOCTYPE html>
<html lang="pt-br">
<head>
<meta charset="utf-8">
<title>Movie Ratings Explorer</title>
<style>
body {
    background: #090d18;
    color: #f1f5f9;
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 32px;
}
header {
    max-width: 1350px;
    margin: auto;
    padding: 25px 0 35px;
}
h1 { font-size: 38px; margin-bottom: 8px; }
header p { color: #94a3b8; font-size: 17px; }
.grafico {
    max-width: 1350px;
    margin: 22px auto;
    padding: 15px;
    background: #111827;
    border: 1px solid #253047;
    border-radius: 16px;
}
h2 { margin: 15px 20px; font-size: 21px; }
</style>
</head>
<body>
<header>
<h1>Movie Ratings Explorer</h1>
<p>Uma exploração multidimensional de usuários, filmes e avaliações</p>
</header>
""" + "\n".join(partes) + """
</body>
</html>
"""

saida = Path("movie_ratings_dashboard.html")
saida.write_text(html, encoding="utf-8")

print(f"Dashboard criado: {saida.resolve()}")
display(fig1)


Dashboard criado: /Users/insider/Desktop/movie-rating-prediction-main/notebooks/movie_ratings_dashboard.html
